# Volume 1, Chapter 1: Divisibility and the Euclidean Algorithm (Python companion)

Exercises first, solutions at the end. Computer-science threads: the **extended Euclidean algorithm** (modular inverses,
the workhorse of RSA), **worst-case analysis** with Lamé, and **binary GCD**, the version CPUs prefer.

In [ ]:
import math
import random
import time
import matplotlib.pyplot as plt
# Reproducible teaching examples; these seeds are not for cryptographic keys.
import random
from sympy.core.random import seed as seed_sympy
random.seed(10101)
seed_sympy(20101)


## Before the experiment: Keep Euclid honest

Before running the solutions, predict the Bézout coefficients for $91$ and $35$. A gcd is unique; are its coefficients? Try replacing $(u,v)$ by $(u+35/g,v-91/g)$. The algorithm carries an invariant, not a rabbit hidden in a remainder.

**How to use this notebook.** Try the exercise cells first; they are intentionally blank. Then run the solutions in order in a fresh kernel. Assertions check exact identities or a stated finite range; plots and random trials supply evidence, not proofs.


## Exercises

**P1.1 (★) Extended Euclid.** Implement `egcd(a, b)` returning $(g, u, v)$ with $au + bv = g = \gcd(a,b)$, iteratively as in Section 1.2. Use it to write `inverse(a, n)` (the inverse of $a$ modulo $n$, when $\gcd(a,n) = 1$), and compare with Python's built-in `pow(a, -1, n)`. Recover the solution of Exercise 1.1.

In [ ]:
# Your code here


**P1.2 (★) Lamé, observed.** Count the division steps of Euclid's algorithm for all pairs $1 \le b < a \le 1000$. Which pairs need the most steps? Plot the maximum number of steps for $b < N$ against $\log_\varphi N$.

In [ ]:
# Your code here


**P1.3 (★★) The coin problem by machine.** Check $ab - a - b$ for all coprime $a, b \le 30$ by brute force. Then explore three coins: compute the Frobenius number for $\{6, 9, 20\}$ (the original *McNugget* boxes).

In [ ]:
# Your code here


**P1.4 (★★) CS application: binary GCD.** Stein's algorithm avoids divisions: $\gcd(2a, 2b) = 2\gcd(a,b)$, $\gcd(2a, b) = \gcd(a, b)$ for $b$ odd, and $\gcd(a, b) = \gcd(|a-b|, \min(a,b))$ for $a, b$ odd. Implement it with bit operations, test it against `math.gcd`, and count its iterations against Euclid's on random 1000-bit numbers.

In [ ]:
# Your code here


**P1.5 (★★) Legendre and the super-Catalan numbers.** Write `v_p_factorial(n, p)` with Legendre's formula, check it against direct factorisation for $n \le 300$, and verify that $\frac{(2m)!(2n)!}{m!\,n!\,(m+n)!}$ is an integer for $m, n \le 40$ (Exercise 1.8).

In [ ]:
# Your code here


---
## Solutions

### P1.1

In [ ]:
def egcd(a, b):
    u0, v0, u1, v1 = 1, 0, 0, 1
    while b:
        q, r = divmod(a, b)
        a, b = b, r
        u0, u1 = u1, u0 - q * u1
        v0, v1 = v1, v0 - q * v1
    if a < 0:
        a, u0, v0 = -a, -u0, -v0
    return a, u0, v0

def inverse(a, n):
    if n < 2:
        raise ValueError("the modulus must be at least 2")
    g, u, _ = egcd(a % n, n)
    if g != 1:
        raise ValueError(f"{a} is not invertible modulo {n}")
    return u % n

print(egcd(91, 35), egcd(240, 46))
for _ in range(10_000):
    n = random.randrange(2, 10**30)
    a = random.randrange(1, n)
    if math.gcd(a, n) == 1:
        assert inverse(a, n) == pow(a, -1, n)
print("inverse() agrees with pow(a, -1, n)")
for a in range(-20, 21):
    for b in range(-20, 21):
        g, u, v = egcd(a, b)
        assert g == math.gcd(a, b) == a*u + b*v


### P1.2

In [ ]:
def steps(a, b):
    s = 0
    while b:
        a, b = b, a % b
        s += 1
    return s

best = max(((steps(a, b), a, b) for a in range(2, 1001) for b in range(1, a)), key=lambda t: t[0])
print("most steps for a <= 1000:", best, "(consecutive Fibonacci numbers)")

phi = (1 + 5 ** 0.5) / 2
Ns = [2**k for k in range(2, 40)]
fib = [1, 2]
while fib[-1] < 2**40:
    fib.append(fib[-1] + fib[-2])
worst = [max(steps(fib[i + 1], fib[i]) for i in range(len(fib) - 1) if fib[i] < N) for N in Ns]
plt.plot([math.log(N, phi) for N in Ns], worst, "o-")
plt.xlabel(r"$\log_\varphi N$"); plt.ylabel("max steps with b < N"); plt.title("Lamé: worst case grows like log_phi N"); plt.show()

### P1.3

In [ ]:
def frobenius(coins, limit=10_000):
    ok = [False] * (limit + 1)
    ok[0] = True
    for m in range(1, limit + 1):
        ok[m] = any(m >= c and ok[m - c] for c in coins)
    # A run of min(coins) representable integers guarantees all later ones.
    if not all(ok[limit - min(coins) + 1:]):
        raise ValueError("increase limit: the finite search has not certified the tail")
    return max((m for m in range(limit + 1) if not ok[m]), default=-1)

for a in range(1, 31):
    for b in range(1, 31):
        if math.gcd(a, b) == 1 and a > 1 and b > 1:
            assert frobenius([a, b], 1000) == a * b - a - b
print("ab - a - b confirmed for coprime a, b <= 30")
print("Frobenius number of {6, 9, 20}:", frobenius([6, 9, 20]))

### P1.4

In [ ]:
def binary_gcd(a, b):
    a, b = abs(a), abs(b)
    if a == 0 or b == 0:
        return a | b, 0
    shift = ((a | b) & -(a | b)).bit_length() - 1      # common power of 2
    a >>= (a & -a).bit_length() - 1
    it = 0
    while b:
        b >>= (b & -b).bit_length() - 1
        if a > b:
            a, b = b, a
        b -= a
        it += 1
    return a << shift, it

for _ in range(10_000):
    x, y = random.getrandbits(64), random.getrandbits(64)
    assert binary_gcd(x, y)[0] == math.gcd(x, y)

x, y = random.getrandbits(1000), random.getrandbits(1000)
print("Euclid steps:", steps(x, y), "  binary GCD iterations:", binary_gcd(x, y)[1])

Binary GCD replaces division by shifts and subtraction. An iteration count is not a running-time comparison: Euclid and binary GCD perform different operations, and the balance depends on the integer size and implementation. Test timings separately before making a speed claim. An algorithm can win the step-count race while wearing much heavier shoes.


### P1.5

In [ ]:
from sympy import factorint

def v_p_factorial(n, p):
    v, q = 0, p
    while q <= n:
        v += n // q
        q *= p
    return v

for n in range(2, 301):
    f = factorint(math.factorial(n))
    assert all(v_p_factorial(n, p) == e for p, e in f.items())
for m in range(41):
    for n in range(41):
        assert (math.factorial(2 * m) * math.factorial(2 * n)) % (math.factorial(m) * math.factorial(n) * math.factorial(m + n)) == 0
print("Legendre's formula and the super-Catalan integrality confirmed;", "trailing zeros of 1000! =", v_p_factorial(1000, 5))

## Euclid's fingerprint

Predict where a heat map of division counts will be dark. The diagonal itself is easy; the difficult inputs lie near ratios with a long continued fraction. Here color means a precisely defined count, not elapsed time.


In [ ]:
grid = [[steps(a, b) for b in range(1, 101)] for a in range(1, 101)]
plt.figure(figsize=(6, 5))
plt.imshow(grid, origin="lower", extent=(0.5, 100.5, 0.5, 100.5), cmap="cividis", aspect="equal")
plt.colorbar(label="Euclidean divisions (including the final zero remainder)")
plt.xlabel("b"); plt.ylabel("a"); plt.title("Euclid's algorithm: exact step counts")
plt.tight_layout(); plt.show()


## One more question before closing

Find a pair for which Euclid and binary GCD disagree in iteration count. Explain why the counts cannot directly decide which is faster.

Write one sentence identifying what was proved in the chapter, what was checked exactly by the code, and what remains a conjecture or a heuristic.
